# Chain-of-Thought & Structured Output

Companion notebook for the [Chain-of-Thought lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/02-chain-of-thought).

**The idea in one sentence.** Two reliability techniques: **chain-of-thought** lets the
model spend tokens reasoning step by step (and **self-consistency** samples several chains
and takes the majority vote), while **structured output** forces the answer into a
machine-parseable schema so you *parse, don't hope*.

The mechanics:

- **Self-consistency:** each reasoning chain is a noisy vote for an answer; the majority
  of several chains is more accurate than one (a Condorcet effect).
- **Structured output:** validate the model's JSON against a schema and reject
  free-form prose.

We build both, **validate that self-consistency improves accuracy and that schema
validation rejects malformed output**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Self-consistency by majority vote

A single reasoning chain can slip. If we sample several **independent** chains and the correct answer is more probable than any single wrong one, majority voting boosts accuracy. We simulate a noisy reasoner whose chains are right 55% of the time.

In [ ]:
rng = np.random.default_rng(0)
TRUE = 42
WRONG = [7, 13, 99]

def sample_chain():
    # 55% chance to reach the true answer, else a random wrong one
    return TRUE if rng.random() < 0.55 else int(rng.choice(WRONG))

from collections import Counter
def answer_with_k(k):
    votes = Counter(sample_chain() for _ in range(k))
    return votes.most_common(1)[0][0]

for k in [1, 3, 5, 11]:
    acc = np.mean([answer_with_k(k) == TRUE for _ in range(2000)])
    print(f'k={k:2d} chains -> accuracy {acc:.3f}')

### Validate: self-consistency (majority vote) beats a single chain

Each chain reaches the true answer 55% of the time — better than chance but unreliable
alone. Taking the majority of several chains raises accuracy (Condorcet's jury theorem:
independent, better-than-chance voters improve with count). We confirm accuracy rises
with the number of chains.

In [ ]:
accs = {}
for k in [1, 3, 5, 11]:
    accs[k] = np.mean([answer_with_k(k) == TRUE for _ in range(3000)])
    print(f'k={k:2d} chains -> accuracy {accs[k]:.3f}')
assert accs[11] > accs[1], 'more chains -> higher accuracy (self-consistency)'
assert accs[1] > 0.5, 'a single chain is better than chance but unreliable'
print('\n✅ self-consistency: the majority of several reasoning chains beats one chain')

More chains → higher accuracy, with diminishing returns — the cost is running the model `k` times.

## Structured output: parse, don't hope

Downstream code needs machine-readable output. Asking for JSON in prose is fragile; validating against a schema is robust. Here we contrast brittle regex extraction with a schema check.

In [ ]:
import json

good = '{"category": "billing", "confidence": 0.91}'
bad  = 'Sure! Here is the answer: category=billing (91% sure)'

SCHEMA = {'category': str, 'confidence': float}
def validate(text):
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return None
    if all(k in obj and isinstance(obj[k], t) for k, t in SCHEMA.items()):
        return obj
    return None

print('good ->', validate(good))
print('bad  ->', validate(bad))

### Validate: schema validation parses well-formed output and rejects prose

The whole point of structured output is that you *parse* the answer, not hope it's
usable. We confirm the validator accepts schema-conforming JSON and rejects both
free-form prose and JSON with the wrong types — the contract your downstream code
relies on.

In [ ]:
assert validate(good) is not None, 'valid JSON matching the schema should parse'
assert validate(bad) is None, 'free-form prose should be rejected'
assert validate('{"category": "billing", "confidence": "high"}') is None, 'wrong type should be rejected'
assert validate('{"category": "billing"}') is None, 'missing field should be rejected'
print('good JSON     ->', validate(good))
print('prose         ->', validate(bad))
print('wrong types   ->', validate('{"category": "billing", "confidence": "high"}'))
print('\n✅ parse, don\'t hope: schema validation admits conforming JSON and rejects the rest')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **self-consistency below 50%** | majority vote makes a weak model *worse* (demo) |
| **cost of many chains** | k chains ≈ k× tokens; diminishing returns past a handful |
| **hoping instead of parsing** | free-form output breaks downstream code; validate the schema |
| **CoT isn't always better** | on simple tasks it adds cost and can introduce errors |
| **leaking the reasoning** | expose the final answer, not always the chain, to users |

Demo: self-consistency helps only when the chains are better than chance.

In [ ]:
# Why self-consistency works (and when it doesn't): it needs chains that are INDEPENDENT
# and better-than-chance. If per-chain accuracy drops below 0.5, majority vote gets WORSE
# with more chains — the same Condorcet knife-edge as multi-agent voting.
rng2 = np.random.default_rng(1)
def vote_accuracy(p_correct, k, trials=3000):
    def one():
        votes = Counter(TRUE if rng2.random() < p_correct else int(rng2.choice(WRONG)) for _ in range(k))
        return votes.most_common(1)[0][0]
    return np.mean([one() == TRUE for _ in range(trials)])
for p in [0.6, 0.4]:
    a1, a11 = vote_accuracy(p, 1), vote_accuracy(p, 11)
    print(f'per-chain accuracy {p}: k=1 -> {a1:.2f}, k=11 -> {a11:.2f}')
print('\nAbove 0.5 more chains help; below 0.5 they HURT -> self-consistency needs competent chains.')

## ✏️ Your turn

Implement `majority_vote(answers)` returning the most common element (the core of self-consistency).

In [ ]:
def majority_vote(answers):
    # TODO(you): return the value that appears most often in `answers`.
    return None

assert majority_vote([42, 7, 42, 99, 42]) == 42
assert majority_vote(['a', 'b', 'b']) == 'b'
print('passed ✓')

<details><summary>Solution</summary>

```python
from collections import Counter
def majority_vote(answers):
    return Counter(answers).most_common(1)[0][0]
```

</details>

## Key takeaways

- **Chain-of-thought spends tokens on reasoning;** self-consistency samples several
  chains and votes.
- **Self-consistency improves accuracy** when chains are independent and
  better-than-chance (verified) — but *hurts* below 50% per-chain accuracy (demo).
- **Structured output = parse, don't hope:** validate against a schema and reject
  malformed output (verified) — the contract downstream code depends on.
- **Reasoning costs tokens:** more chains and longer CoT trade cost for reliability.